# CNV anthropometry (Task A probe GWAS + Task B ConGA axes) - All of Us Researcher Workbench
Same code base as the UKBB/Expanse run; only `configs/aou_workbench.json` differs. Controlled Tier, Python kernel.

Individual-level files stay on this persistent disk / the workspace bucket. Only `export_aggregate/` may be downloaded, after the 1-20 small-cell rule (applied by the export step).

Axis definitions: ConGA (Engchuan et al., companion manuscript).

In [ ]:
import os, subprocess, json
print(os.environ.get('WORKSPACE_CDR'), os.environ.get('WORKSPACE_BUCKET'), os.environ.get('GOOGLE_PROJECT'))
CODE = os.path.expanduser('~/cnv_anthro_code')   # copy cnv_anthro/ and resources/ here (e.g. gsutil -m cp -r ${WORKSPACE_BUCKET}/cnv_anthro_code ~/)
os.makedirs(os.path.expanduser('~/cnv_anthro_run/data'), exist_ok=True); os.makedirs(os.path.expanduser('~/cnv_anthro_run/results'), exist_ok=True)

In [ ]:
# packages used: numpy, scipy, pandas, matplotlib, google-cloud-bigquery (all preinstalled in the Workbench Python image)
import numpy, scipy, pandas, matplotlib
from google.cloud import bigquery
print(numpy.__version__, scipy.__version__, pandas.__version__)

## 1. Fill the two data paths
From *How the All of Us Genomic data are organized* (the version matching this workspace's CDR): the srWGS **SV joint-called VCF** and the **ancestry_preds.tsv**. Copy the ancestry file locally; the VCF is streamed with `gsutil cat` (requester-pays project = GOOGLE_PROJECT).

In [ ]:
cfg_path = f'{CODE}/cnv_anthro/configs/aou_workbench.json'
cfg = json.load(open(cfg_path))
SV_VCF = 'gs://...'          # <- fill
ANCESTRY_GS = 'gs://...'     # <- fill
subprocess.run(['gsutil', '-u', os.environ['GOOGLE_PROJECT'], 'cp', ANCESTRY_GS, os.path.expanduser('~/cnv_anthro_run/data/ancestry_preds.tsv')], check=True)
cfg['prep_commands'][0][2] = SV_VCF
cfg['prep_commands'][1][2] = os.path.expanduser('~/cnv_anthro_run/data/ancestry_preds.tsv')
json.dump(cfg, open(cfg_path, 'w'), indent=1)

## 2. Prep (SV VCF -> CNV table; OMOP -> phenotype table)
The SV VCF pass is the slowest step (one streaming pass over ~97k samples); run it on a large VM or with dsub (`workbench/dsub_run.sh`).

In [ ]:
!cd {CODE}/cnv_anthro && python run_pipeline.py --config configs/aou_workbench.json --steps prep
print(open(os.path.expanduser('~/cnv_anthro_run/results/aou_pheno_summary.json')).read())

## 3. Task A - probe-level CNV GWAS + allele tests (height, BMI)

In [ ]:
!cd {CODE}/cnv_anthro && python run_pipeline.py --config configs/aou_workbench.json --steps gwas,plots,alleles

## 4. Task B - ConGA axes with genome-shift null, 18 LOO and no-loci runs
About 3-4 h per trait x panel set at 2,000 replicates on 4 cores; parallelise with `--only_trait` / `--only_panels` (dsub tasks).

In [ ]:
!cd {CODE}/cnv_anthro && python run_pipeline.py --config configs/aou_workbench.json --steps axes --only_trait height --only_panels gtex_nonbrain

## 5. Export aggregate results (small-cell rule enforced)

In [ ]:
!cd {CODE}/cnv_anthro && python run_pipeline.py --config configs/aou_workbench.json --steps export
!ls -R ~/cnv_anthro_run/export_aggregate | head -50